In [10]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

## Exercise 1: Sparse optical flow

In [11]:
cap = cv2.VideoCapture('Robots.mp4')
cv2.namedWindow('output_frame', cv2.WINDOW_NORMAL)

In [12]:
ret, old_frame = cap.read()
b,g,r = cv2.split(old_frame) # Changing the order from bgr to rgb so that matplotlib can show it
old_frame = cv2.merge([r,g,b])
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_RGB2GRAY)
#plt.imshow(old_gray, cmap = 'gray')
old_feat = cv2.goodFeaturesToTrack(old_gray, maxCorners=300, qualityLevel=0.3, minDistance=7)
mask = np.zeros_like(old_frame)
while True:
    ret, new_frame = cap.read()
    if not ret:
        break
    b,g,r = cv2.split(new_frame) # Changing the order from bgr to rgb so that matplotlib can show it
    new_frame = cv2.merge([r,g,b])
    new_gray = cv2.cvtColor(new_frame, cv2.COLOR_RGB2GRAY)
    #plt.imshow(new_gray, cmap = 'gray')
    new_feat, status, error = cv2.calcOpticalFlowPyrLK(old_gray, new_gray, old_feat, None)
    if new_feat is not None:
        valid_new = []
        for i in range(len(old_feat)):
            if status[i] == 1:
                f10 = int(old_feat[i][0][0])
                f11 = int(old_feat[i][0][1])
                f20 = int(new_feat[i][0][0])
                f21 = int(new_feat[i][0][1])
                mask = cv2.line(mask, (f10, f11), (f20, f21), (0, 255, 0), 2)
                new_frame = cv2.circle(new_frame, (f20, f21), 5, (255, 0, 0), -1)
                valid_new.append(new_feat[i])
                
        
        output_frame = cv2.add(new_frame, mask)
        old_frame=new_frame
        old_gray=new_gray
        old_feat=np.array(valid_new)
        cv2.imshow('output_frame', output_frame)
        cv2.waitKey(20)
    else:
        break

cv2.destroyAllWindows()
cv2.waitKey(1)


-1

## Exercise 2:  Dense optical flow

### Helper functions for different displays of the dense optical flow

In [13]:
def dense_grid(mag, ang, frame, alpha=0.3):

    mag = mag*3
        
    sampling = 10

    sam_mag, sam_ang = mag[::sampling, ::sampling], ang[::sampling, ::sampling]

    empty = np.zeros(shape=(mag.shape[0], mag.shape[1], 3), dtype=np.uint8)

    # build the start grid from the actual frame size
    start = np.mgrid[0:mag.shape[0]:sampling, 0:mag.shape[1]:sampling]

    end_point = (
        np.clip(np.squeeze(start[0, ...]) + sam_mag * np.cos(sam_ang), 0, mag.shape[0]),
        np.clip(np.squeeze(start[1, ...]) - sam_mag * np.sin(sam_ang), 0, mag.shape[1])
    )

    for i in range(sam_mag.shape[0]):
        for j in range(sam_mag.shape[1]):
            cv2.line(empty, (j * sampling, i * sampling), (int(end_point[1][i, j]), int(end_point[0][i, j])), (0, 255, 0), 2)
            cv2.circle(empty, (int(end_point[1][i, j]), int(end_point[0][i, j])), 3, (0, 0, 255), -1)

    #flow_bgr = cv2.cvtColor(empty, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(empty, alpha, frame, 1 - alpha, 0)
    return overlay


def dense_color_repr(mag, ang, frame, alpha=0.6):
    hsv = np.zeros((*mag.shape, 3), dtype=np.uint8)
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

    flow_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # blend: alpha*flow + (1-alpha)*original frame
    overlay = cv2.addWeighted(flow_bgr, alpha, frame, 1 - alpha, 0)
    return overlay

### Loop showing the dense optical flow video

In [ ]:
cap = cv2.VideoCapture('Robots.mp4') # Reload the video

ret = True
previous_frame = np.array(0)
frame_to_show = np.array(0)

while ret:
        ret, current_frame = cap.read() # Read every frame
        if not ret: 
                break # break if not read
        
        # Hold current frame
        if previous_frame.any():
                # Convert to grayscale
                gray1 = cv2.cvtColor(previous_frame, cv2.COLOR_RGB2GRAY)
                gray2 = cv2.cvtColor(current_frame, cv2.COLOR_RGB2GRAY)

                # Get the flow between the pictures but downscale to make it run faster
                scale = 0.25  #
                small1 = cv2.resize(gray1, None, fx=scale, fy=scale)
                small2 = cv2.resize(gray2, None, fx=scale, fy=scale)

                flow = cv2.calcOpticalFlowFarneback(small1, small2, None, 0.5, 3, 15, 3, 5, 1.2, 0)

                # scale flow vectors back up and resize the flow field to full res
                flow = cv2.resize(flow, (gray1.shape[1], gray1.shape[0]))
                flow *= 1 / scale

                mag, ang = cv2.cartToPolar(flow[:,:,0], flow[:,:,1])# Convert polar to cartesian

                # Choose how to show the dense optical flow
                frame_to_show = dense_color_repr(mag, ang, current_frame)       # Color version
                #frame_to_show = dense_grid(mag, ang, current_frame)            # Grid-version showing the dense optical flow
        else:
                frame_to_show = current_frame   # nothing to compare yet, just show the raw frame

        cv2.imshow('image', frame_to_show)
        cv2.waitKey(20)

        previous_frame = current_frame

        # Break the loop by typing "q"       
        if cv2.waitKey(20) & 0xFF == ord('q'):
                break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)   # let the event loop actually process the close

-1

The dense optical flow is shown with a similar loop. Instead of now tracking the features and comparing them, we instead use the `calcOpticalFlowFarneback` function. This is quite heavy, and at first, the video was playing very slowly, so we ended up downscaling it to make the video run more smoothly.

Then we displayed the flow in 2 ways:

1. `dense_color_repr`: With an overlay of colors, where the color represent the direction and the intensity represent the magnitude.
2. `dense_grid`: An overlay like shown in the solution for week2 exercise4 where the direction is shown with green arrows.